# Example 02 - 1D PLUMED FES + Constant D

This notebook mirrors `examples/02_1d_plumed_fes_ctmc.py` using the
bundled synthetic 1D FES. It runs the high-level 1D CTMC workflow,
inspects the rate matrix, and plots the basin partitioning.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "stochkin").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "examples" / "data"
OUT_DIR = ROOT / "notebooks" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {ROOT}")
print(f"Notebook output directory: {OUT_DIR}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import stochkin as sk

from stochkin.plotting import _apply_pub_axes, _apply_pub_cbar
from stochkin.style import LEGEND_SIZE, publication_style


In [ ]:
fes_path = DATA_DIR / "synthetic_1d_fes.dat"
D = 0.05
crop = (0.5, 9.5)
T = 300.0
resample_n = 500
core_fraction = 0.05

result = sk.run_1d_ctmc_from_plumed(
    fes_path=fes_path,
    D=D,
    T=T,
    crop=crop,
    resample_n=resample_n,
    core_fraction=core_fraction,
)

print(f"Basins: {result['basin_ids']}")


In [ ]:
basin_labels = [f"B{bid}" for bid in result["basin_ids"]]
K_ns = pd.DataFrame(
    result["K_ps"] * 1000.0,
    index=basin_labels,
    columns=basin_labels,
)
exit_times_ns = pd.Series(
    result["exit_ps"] / 1000.0,
    index=basin_labels,
    name="exit_time_ns",
)

K_ns


In [ ]:
exit_times_ns


In [ ]:
s = result["s"]
F = result["F"]
labels = result["labels_full"]
output_path = OUT_DIR / "02_ctmc_rates.png"

with publication_style():
    cmap = plt.get_cmap("tab10")
    fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.8))

    ax = axes[0]
    ax.plot(s, F, "k-", lw=1.5)
    for bid in result["basin_ids"]:
        mask = labels == bid
        ax.fill_between(
            s,
            F,
            where=mask,
            alpha=0.25,
            color=cmap(int(bid)),
            label=f"B{bid}",
        )
    _apply_pub_axes(ax, "CV", r"F  [kJ mol$^{-1}$]", "FES + basins")
    ax.legend(fontsize=LEGEND_SIZE, frameon=False)

    ax = axes[1]
    K_abs = np.abs(K_ns.to_numpy())
    with np.errstate(divide="ignore", invalid="ignore"):
        Klog = np.where(K_abs > 0, np.log10(K_abs), np.nan)
    im = ax.imshow(Klog, cmap="magma_r", aspect="auto")
    ax.set_xticks(range(len(basin_labels)))
    ax.set_xticklabels(basin_labels)
    ax.set_yticks(range(len(basin_labels)))
    ax.set_yticklabels(basin_labels)
    cbar = fig.colorbar(im, ax=ax)
    _apply_pub_cbar(cbar, label=r"$\log_{10}|K_{ij}|$  [ns$^{-1}$]")
    _apply_pub_axes(ax, title="Rate matrix")

    fig.tight_layout()
    fig.savefig(output_path, dpi=300)

print(f"Saved {output_path.relative_to(ROOT)}")
plt.show()
